# AISC DeepFake — Learnable Logistic Region Fusion

Bu notebook, aynı model ailesinin **Eye + Brow + Mouth** bölgesel fake
olasılıklarını küçük ve açıklanabilir bir **Logistic Regression meta-modeli**
ile birleştirir.

## Model families

1. **Swin V2 Tiny**
2. **EfficientNet-B0**
3. **Swin V2 Tiny + Texture Fusion**

## Fusion input

Her ortak frame için meta-model yalnızca üç girdi görür:

```text
[p_eye, p_brow, p_mouth]
```

ve tek bir final fake olasılığı üretir.

## Scientific design

- Base detector'lar yeniden eğitilmez.
- Logistic fusion **TEST üzerinde asla fit edilmez**.
- Meta-modelin eğitim verisi, üç bölgenin ortak **validation prediction**
  kesişimidir.
- `StandardScaler` yalnız validation/meta-training verisinde fit edilir.
- Logistic Regression hiperparametreleri önceden sabittir (`C=1.0`);
  test setine bakılarak ayarlanmaz.
- Ana karar eşiği yöntemler arası karşılaştırma için **0.50** olarak önceden
  sabittir.
- Validation içerisinde Stratified K-Fold OOF değerlendirmesi yapılır; bu
  yalnız bir sanity/generalization kontrolüdür.
- Final meta-model validation'ın tamamında fit edilir ve test seti yalnız
  final değerlendirmede kullanılır.
- Validation prediction dosyaları bulunamazsa notebook **fail-fast** durur.
  TEST verisini meta-training verisi olarak kullanarak sonucu yapay biçimde
  iyileştirmez.

## Project/code standards applied

Bu sürümde read-only kaynak veri, seed/reproducibility, açık veri muhasebesi,
schema quality gates, cache/hash provenance çözümü, train/test izolasyonu,
atomik kayıt, benzersiz Run ID, inference quality gates, İngilizce grafikler,
600-DPI PNG + SVG çıktılar ve final output manifest uygulanır.

> **Methodological note:** Base modeller validation setini early stopping/model
> selection için görmüş olabilir. Bu nedenle learnable fusion için en güçlü
> tasarım ayrı bir meta-training split veya out-of-fold base predictions olur.
> Bu notebook mevcut deney yapısında test sızıntısını engelleyen pratik
> tasarımı kullanır ve bu sınırlılığı audit çıktısına kaydeder.

In [1]:
# ============================================================
# 1) COLAB + CONFIG
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version

import json
import math
import os
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

METHOD_NAME = "03_learnable_logistic_fusion"
DECISION_THRESHOLD = 0.50
FRAME_AGGREGATION = "mean"

LOGISTIC_C = 1.0
LOGISTIC_MAX_ITER = 5000
LOGISTIC_CLASS_WEIGHT = "balanced"

FIGURE_DPI = 600
MIN_FIGURE_SHORT_EDGE_PX = 600

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "AISC DeepFake Çalışmaları/Deney 1/"
    "Kader/Deney 1/Sonuçlar/Fusion_Experiments"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SEARCH_ROOTS = {
    "eye": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Kader/Deney 1/Sonuçlar"
    ),
    "brow": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Nazlıcan/Deney 1/Sonuçlar"
    ),
    "mouth": Path(
        "/content/drive/MyDrive/"
        "AISC DeepFake Çalışmaları/Deney 1/"
        "Dilara/Deney 1/Sonuçlar"
    ),
}

# ------------------------------------------------------------------
# TEST prediction paths already verified in the other fusion notebooks.
# Source files are READ ONLY.
# ------------------------------------------------------------------

MODEL_FAMILIES = {
    "swinv2_tiny": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260807_1031_eye_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260807_2235_mouth_swinv2_tiny_seed42"
            / "predictions"
            / "test_frame_predictions.csv"
        ),
    },

    "efficientnet_b0": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260808_0803_eye_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "20260808_1248_eyebrow_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "20260808_1257_mouth_efficientnet_b0_seed42"
            / "predictions"
            / "test_predictions.csv"
        ),
    },

    "swinv2_texture": {
        "eye_test": (
            SEARCH_ROOTS["eye"]
            / "20260806_1748_eye_swinv2_texturefusion_seed42"
            / "full"
            / "predictions"
            / "test_predictions.csv"
        ),
        "brow_test": (
            SEARCH_ROOTS["brow"]
            / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion"
            / "predictions"
            / "test_predictions_frame_level.csv"
        ),
        "mouth_test": (
            SEARCH_ROOTS["mouth"]
            / "SwinV2_TextureFusion_Mouth"
            / "20260807_1550_mouth_swinv2_texturefusion_seed42"
            / "full"
            / "predictions"
            / "test_predictions.csv"
        ),
    },
}

# ------------------------------------------------------------------
# Optional manual validation-prediction paths.
#
# Leave None: notebook searches the corresponding experiment directory.
# Fill only with a TRUE validation prediction file if automatic discovery
# cannot find it.
#
# Never put a TEST prediction file here.
# ------------------------------------------------------------------

VALIDATION_PREDICTION_OVERRIDE = {
    family: {
        "eye": None,
        "brow": None,
        "mouth": None,
    }
    for family in MODEL_FAMILIES
}

RUN_ID = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%d_%H%M%S_%f")
    + "_learnable_logistic_fusion_seed42"
)

RUN_DIR = (
    OUTPUT_ROOT
    / METHOD_NAME
    / RUN_ID
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

print(f"Run ID     : {RUN_ID}")
print(f"Run output : {RUN_DIR}")

Mounted at /content/drive
Run ID     : 20260809_164437_273718_learnable_logistic_fusion_seed42
Run output : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/03_learnable_logistic_fusion/20260809_164437_273718_learnable_logistic_fusion_seed42


In [2]:
# ============================================================
# 2) ENVIRONMENT + ATOMIC I/O
# ============================================================

import joblib
import matplotlib
import matplotlib.pyplot as plt

from PIL import Image


def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not_installed"


def json_default(value):
    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, Path):
        return str(value)

    raise TypeError(
        f"Object of type {type(value).__name__} "
        "is not JSON serializable."
    )


def atomic_write_json(payload, target):
    target = Path(target)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    with temp_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            payload,
            file_handle,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        )
        file_handle.flush()
        os.fsync(
            file_handle.fileno()
        )

    with temp_path.open(
        "r",
        encoding="utf-8",
    ) as file_handle:
        json.load(
            file_handle
        )

    os.replace(
        temp_path,
        target,
    )


def atomic_write_csv(dataframe, target):
    target = Path(target)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    dataframe.to_csv(
        temp_path,
        index=False,
    )

    verification = pd.read_csv(
        temp_path
    )

    if len(verification) != len(dataframe):
        raise RuntimeError(
            f"Atomic CSV verification failed for {target}. "
            f"Expected {len(dataframe)} rows, "
            f"read back {len(verification)}."
        )

    os.replace(
        temp_path,
        target,
    )


def atomic_dump_joblib(model, target):
    target = Path(target)
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    joblib.dump(
        model,
        temp_path,
    )

    # Integrity/inference reload test.
    loaded = joblib.load(
        temp_path
    )

    if not hasattr(
        loaded,
        "predict_proba",
    ):
        raise RuntimeError(
            f"Reloaded fusion model has no predict_proba(): {target}"
        )

    os.replace(
        temp_path,
        target,
    )


def save_figure(fig, target_stem):
    target_stem = Path(
        target_stem
    )

    target_stem.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    png_path = target_stem.with_suffix(
        ".png"
    )
    svg_path = target_stem.with_suffix(
        ".svg"
    )

    temp_png = png_path.with_suffix(
        ".png.tmp"
    )
    temp_svg = svg_path.with_suffix(
        ".svg.tmp"
    )

    fig.savefig(
        temp_png,
        format="png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    fig.savefig(
        temp_svg,
        format="svg",
        bbox_inches="tight",
    )

    with Image.open(
        temp_png
    ) as image:
        if min(
            image.size
        ) < MIN_FIGURE_SHORT_EDGE_PX:
            raise RuntimeError(
                "Figure resolution is insufficient: "
                f"{image.size} -> {png_path}"
            )

    os.replace(
        temp_png,
        png_path,
    )

    os.replace(
        temp_svg,
        svg_path,
    )

    plt.close(
        fig
    )

    return png_path, svg_path


ENVIRONMENT = {
    "run_id": RUN_ID,
    "created_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit_learn": package_version(
        "scikit-learn"
    ),
    "joblib": package_version(
        "joblib"
    ),
    "pillow": package_version(
        "Pillow"
    ),
    "seed": SEED,
    "method": METHOD_NAME,
    "decision_threshold": DECISION_THRESHOLD,
    "frame_aggregation": FRAME_AGGREGATION,
    "logistic_C": LOGISTIC_C,
    "logistic_class_weight": LOGISTIC_CLASS_WEIGHT,
    "logistic_max_iter": LOGISTIC_MAX_ITER,
    "meta_training_source": (
        "aligned validation predictions only"
    ),
    "test_usage": (
        "final evaluation only"
    ),
}

atomic_write_json(
    ENVIRONMENT,
    RUN_DIR
    / "environment.json",
)

In [3]:
# ============================================================
# 3) PREDICTION SCHEMA + UPSTREAM FRAME ALIGNMENT
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

LABEL_CANDIDATES = [
    "true_label",
    "label",
    "label_int",
    "target",
    "y_true",
    "ground_truth",
    "class_id",
    "true_class",
]

PROB_CANDIDATES = [
    "fake_probability",
    "prob_fake",
    "probability_fake",
    "probability",
    "prob",
    "y_score",
    "score",
    "fake_prob",
    "prediction_probability",
]

SOURCE_KEY_CANDIDATES = [
    "source_frame",
    "relative_frame_path",
    "input_path",
    "input_relative_path",
    "original_frame",
    "orijinal_yol",
    "frame_stem",
    "image_path",
    "path",
]


def first_existing(
    columns,
    candidates,
):
    lower_to_original = {
        str(column).lower(): column
        for column in columns
    }

    for candidate in candidates:
        if (
            candidate.lower()
            in lower_to_original
        ):
            return lower_to_original[
                candidate.lower()
            ]

    return None


def normalize_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(
        value,
        (
            int,
            np.integer,
            float,
            np.floating,
        ),
    ):
        return int(
            float(value)
            >= 0.5
        )

    normalized = (
        str(value)
        .strip()
        .lower()
    )

    if normalized in {
        "1",
        "fake",
        "deepfake",
        "manipulated",
        "sahte",
        "f",
    }:
        return 1

    if normalized in {
        "0",
        "real",
        "original",
        "genuine",
        "gerçek",
        "r",
    }:
        return 0

    try:
        return int(
            float(normalized)
            >= 0.5
        )
    except (
        TypeError,
        ValueError,
    ) as exc:
        raise ValueError(
            f"Label could not be normalized: {value!r}"
        ) from exc


def canonical_frame_key(value):
    """
    Convert region-specific ROI names to the same upstream frame key.

    Examples
    --------
    fake_test_00000__face_00.jpg
    fake_test_00000.jpg
    fake_test_fake_test_00000_face00.png

    -> fake_test_00000

    Unknown formats return None; no guessing.
    """

    if pd.isna(value):
        return None

    text = (
        str(value)
        .replace("\\", "/")
        .strip()
        .lower()
    )

    filename = (
        text
        .split("/")[-1]
    )

    filename = re.sub(
        r"\.(jpg|jpeg|png|bmp|webp|npy)$",
        "",
        filename,
        flags=re.IGNORECASE,
    )

    matches = re.findall(
        r"(real|fake)_(train|test|val|validation)_(\d+)",
        filename,
        flags=re.IGNORECASE,
    )

    if not matches:
        return None

    label, split, frame_number = (
        matches[-1]
    )

    split = split.lower()

    if split == "validation":
        split = "val"

    frame_number = (
        str(
            int(frame_number)
        )
        .zfill(5)
    )

    return (
        f"{label.lower()}_"
        f"{split}_"
        f"{frame_number}"
    )


def find_companion_metadata(
    prediction_path,
):
    """
    Find the experiment metadata that maps cache/hash sample IDs
    back to upstream source frames.
    """

    prediction_path = Path(
        prediction_path
    )

    metadata_names = [
        "eligible_metadata.csv",
        "eligible_metadata_before_cache.csv",
        "metadata_used.csv",
    ]

    candidate_paths = []

    current = (
        prediction_path.parent
    )

    for _ in range(7):
        for metadata_name in (
            metadata_names
        ):
            candidate_paths.append(
                current
                / "artifacts"
                / metadata_name
            )

            candidate_paths.append(
                current
                / metadata_name
            )

        if current.parent == current:
            break

        current = current.parent

    seen = set()

    for candidate in (
        candidate_paths
    ):
        candidate = Path(
            candidate
        )

        if candidate in seen:
            continue

        seen.add(
            candidate
        )

        if candidate.is_file():
            return candidate

    return None


def build_key_from_metadata(
    prediction_df,
    prediction_path,
    region_name,
):
    if (
        "sample_id"
        not in prediction_df.columns
    ):
        raise ValueError(
            f"{region_name}: direct frame key unavailable "
            "and sample_id is missing."
        )

    metadata_path = (
        find_companion_metadata(
            prediction_path
        )
    )

    if metadata_path is None:
        raise FileNotFoundError(
            f"{region_name}: cache/hash prediction path detected, "
            "but no companion metadata was found.\n"
            f"Prediction: {prediction_path}"
        )

    metadata = pd.read_csv(
        metadata_path
    )

    if (
        "sample_id"
        not in metadata.columns
    ):
        raise ValueError(
            f"{region_name}: companion metadata has no sample_id.\n"
            f"Metadata: {metadata_path}"
        )

    source_column = None
    source_keys = None

    for candidate in (
        SOURCE_KEY_CANDIDATES
    ):
        if (
            candidate
            not in metadata.columns
        ):
            continue

        candidate_keys = (
            metadata[candidate]
            .map(
                canonical_frame_key
            )
        )

        valid_fraction = float(
            candidate_keys
            .notna()
            .mean()
        )

        if (
            valid_fraction
            >= 0.95
        ):
            source_column = (
                candidate
            )
            source_keys = (
                candidate_keys
            )
            break

    if source_column is None:
        raise ValueError(
            f"{region_name}: companion metadata has no trustworthy "
            "upstream-frame column.\n"
            f"Columns: {list(metadata.columns)}"
        )

    mapping = pd.DataFrame(
        {
            "sample_id": (
                metadata[
                    "sample_id"
                ]
                .astype(str)
                .str.strip()
            ),
            "fusion_key": (
                source_keys
            ),
        }
    )

    mapping = (
        mapping
        .dropna(
            subset=[
                "fusion_key"
            ]
        )
        .drop_duplicates(
            subset=[
                "sample_id"
            ]
        )
    )

    prediction_ids = (
        prediction_df[
            "sample_id"
        ]
        .astype(str)
        .str.strip()
    )

    temporary = pd.DataFrame(
        {
            "sample_id": (
                prediction_ids
            ),
        }
    )

    temporary = (
        temporary
        .merge(
            mapping,
            on="sample_id",
            how="left",
            validate="many_to_one",
        )
    )

    missing_count = int(
        temporary[
            "fusion_key"
        ]
        .isna()
        .sum()
    )

    if missing_count:
        examples = (
            temporary.loc[
                temporary[
                    "fusion_key"
                ].isna(),
                "sample_id",
            ]
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"{region_name}: {missing_count} prediction sample IDs "
            "could not be mapped to source frames.\n"
            f"Examples: {examples}"
        )

    return (
        temporary[
            "fusion_key"
        ],
        {
            "key_resolution": (
                "companion_metadata"
            ),
            "metadata_path": str(
                metadata_path
            ),
            "metadata_source_column": str(
                source_column
            ),
        },
    )


def resolve_fusion_key(
    raw,
    prediction_path,
    region_name,
):
    direct_candidates = [
        "source_frame",
        "relative_frame_path",
        "frame_path",
        "original_frame",
        "image_path",
        "path",
        "frame_stem",
    ]

    for column in (
        direct_candidates
    ):
        if (
            column
            not in raw.columns
        ):
            continue

        keys = (
            raw[column]
            .map(
                canonical_frame_key
            )
        )

        valid_fraction = float(
            keys
            .notna()
            .mean()
        )

        if (
            valid_fraction
            >= 0.95
        ):
            return (
                keys,
                {
                    "key_resolution": (
                        "prediction_column"
                    ),
                    "key_column": str(
                        column
                    ),
                },
            )

    return (
        build_key_from_metadata(
            prediction_df=raw,
            prediction_path=prediction_path,
            region_name=region_name,
        )
    )


def load_prediction_csv(
    path,
    region_name,
):
    if path is None:
        raise FileNotFoundError(
            f"{region_name}: prediction CSV path is None."
        )

    path = Path(
        path
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"{region_name}: prediction CSV does not exist:\n{path}"
        )

    raw = pd.read_csv(
        path
    )

    if raw.empty:
        raise ValueError(
            f"{region_name}: prediction CSV is empty:\n{path}"
        )

    label_col = first_existing(
        raw.columns,
        LABEL_CANDIDATES,
    )

    prob_col = first_existing(
        raw.columns,
        PROB_CANDIDATES,
    )

    if label_col is None:
        raise ValueError(
            f"{region_name}: label column not found.\n"
            f"Columns: {list(raw.columns)}"
        )

    if prob_col is None:
        raise ValueError(
            f"{region_name}: fake probability column not found.\n"
            f"Columns: {list(raw.columns)}"
        )

    labels = (
        raw[label_col]
        .map(
            normalize_label
        )
    )

    if (
        labels
        .isna()
        .any()
    ):
        raise ValueError(
            f"{region_name}: invalid labels found."
        )

    probabilities = (
        pd.to_numeric(
            raw[
                prob_col
            ],
            errors="coerce",
        )
    )

    if (
        probabilities
        .isna()
        .any()
    ):
        raise ValueError(
            f"{region_name}: invalid probabilities found."
        )

    invalid_range = (
        (probabilities < 0)
        | (probabilities > 1)
    )

    if (
        invalid_range
        .any()
    ):
        raise ValueError(
            f"{region_name}: "
            f"{int(invalid_range.sum())} probabilities "
            "are outside [0, 1]."
        )

    fusion_keys, key_audit = (
        resolve_fusion_key(
            raw=raw,
            prediction_path=path,
            region_name=region_name,
        )
    )

    if (
        fusion_keys
        .isna()
        .any()
    ):
        raise ValueError(
            f"{region_name}: unresolved fusion keys remain."
        )

    if (
        "sample_id"
        in raw.columns
    ):
        raw_identifier = (
            raw["sample_id"]
            .astype(str)
        )

    elif (
        "image_path"
        in raw.columns
    ):
        raw_identifier = (
            raw["image_path"]
            .astype(str)
        )

    elif (
        "path"
        in raw.columns
    ):
        raw_identifier = (
            raw["path"]
            .astype(str)
        )

    else:
        raw_identifier = (
            pd.Series(
                np.arange(
                    len(raw)
                ),
                index=raw.index,
            )
            .astype(str)
        )

    normalized = pd.DataFrame(
        {
            "fusion_key": (
                fusion_keys
            ),
            f"label_{region_name}": (
                labels
                .astype(int)
            ),
            f"p_{region_name}": (
                probabilities
                .astype(float)
            ),
            f"raw_key_{region_name}": (
                raw_identifier
            ),
        }
    )

    grouped = (
        normalized
        .groupby(
            "fusion_key",
            as_index=False,
        )
        .agg(
            **{
                f"label_min_{region_name}": (
                    f"label_{region_name}",
                    "min",
                ),
                f"label_max_{region_name}": (
                    f"label_{region_name}",
                    "max",
                ),
                f"p_{region_name}": (
                    f"p_{region_name}",
                    FRAME_AGGREGATION,
                ),
                f"raw_key_{region_name}": (
                    f"raw_key_{region_name}",
                    "first",
                ),
                f"roi_count_{region_name}": (
                    f"p_{region_name}",
                    "size",
                ),
            }
        )
    )

    inconsistent = (
        grouped[
            f"label_min_{region_name}"
        ]
        != grouped[
            f"label_max_{region_name}"
        ]
    )

    if (
        inconsistent
        .any()
    ):
        raise ValueError(
            f"{region_name}: "
            f"{int(inconsistent.sum())} upstream frames "
            "contain conflicting labels."
        )

    grouped[
        f"label_{region_name}"
    ] = (
        grouped[
            f"label_min_{region_name}"
        ]
        .astype(int)
    )

    grouped = (
        grouped
        .drop(
            columns=[
                f"label_min_{region_name}",
                f"label_max_{region_name}",
            ]
        )
    )

    audit = {
        "region": region_name,
        "prediction_path": str(
            path
        ),
        "source_rows": int(
            len(raw)
        ),
        "unique_upstream_frames": int(
            len(grouped)
        ),
        "multi_roi_frames": int(
            (
                grouped[
                    f"roi_count_{region_name}"
                ]
                > 1
            )
            .sum()
        ),
        "max_roi_count_per_frame": int(
            grouped[
                f"roi_count_{region_name}"
            ]
            .max()
        ),
        "label_column": str(
            label_col
        ),
        "probability_column": str(
            prob_col
        ),
        "frame_aggregation": (
            FRAME_AGGREGATION
        ),
        **key_audit,
    }

    print(
        f"{region_name.upper():5s} | "
        f"Rows={len(raw)} | "
        f"Frames={len(grouped)} | "
        f"Key={key_audit['key_resolution']}"
    )

    return (
        grouped,
        audit,
    )


def align_three(
    eye_path,
    brow_path,
    mouth_path,
):
    eye, eye_audit = (
        load_prediction_csv(
            eye_path,
            "eye",
        )
    )

    brow, brow_audit = (
        load_prediction_csv(
            brow_path,
            "brow",
        )
    )

    mouth, mouth_audit = (
        load_prediction_csv(
            mouth_path,
            "mouth",
        )
    )

    aligned = (
        eye
        .merge(
            brow,
            on="fusion_key",
            how="inner",
            validate="one_to_one",
        )
        .merge(
            mouth,
            on="fusion_key",
            how="inner",
            validate="one_to_one",
        )
    )

    if aligned.empty:
        raise RuntimeError(
            "Eye/Brow/Mouth upstream frame intersection is empty."
        )

    label_columns = [
        "label_eye",
        "label_brow",
        "label_mouth",
    ]

    label_matrix = (
        aligned[
            label_columns
        ]
        .astype(int)
    )

    consistent = (
        label_matrix
        .nunique(axis=1)
        .eq(1)
    )

    if not (
        consistent.all()
    ):
        examples = (
            aligned.loc[
                ~consistent,
                [
                    "fusion_key",
                    *label_columns,
                ],
            ]
            .head(10)
        )

        raise ValueError(
            f"{int((~consistent).sum())} aligned frames have "
            f"inconsistent labels.\n{examples}"
        )

    # Final label is added after consistency check and is NOT dropped.
    aligned["label"] = (
        label_matrix
        .iloc[:, 0]
        .astype(int)
    )

    for column in [
        "p_eye",
        "p_brow",
        "p_mouth",
    ]:
        values = pd.to_numeric(
            aligned[column],
            errors="coerce",
        )

        if (
            values
            .isna()
            .any()
        ):
            raise ValueError(
                f"{column}: NaN probability after alignment."
            )

        if (
            (
                (values < 0)
                | (values > 1)
            )
            .any()
        ):
            raise ValueError(
                f"{column}: probability outside [0, 1]."
            )

        aligned[column] = (
            values
            .astype(float)
        )

    aligned = (
        aligned
        .sort_values(
            "fusion_key"
        )
        .reset_index(
            drop=True
        )
    )

    counts = {
        "eye_frames": int(
            len(eye)
        ),
        "brow_frames": int(
            len(brow)
        ),
        "mouth_frames": int(
            len(mouth)
        ),
        "common_frames": int(
            len(aligned)
        ),
        "real_common_frames": int(
            (
                aligned[
                    "label"
                ]
                == 0
            )
            .sum()
        ),
        "fake_common_frames": int(
            (
                aligned[
                    "label"
                ]
                == 1
            )
            .sum()
        ),
    }

    if (
        counts[
            "real_common_frames"
        ]
        == 0
        or counts[
            "fake_common_frames"
        ]
        == 0
    ):
        raise RuntimeError(
            "Aligned common set must contain both REAL and FAKE."
        )

    return (
        aligned,
        {
            "counts": counts,
            "eye": eye_audit,
            "brow": brow_audit,
            "mouth": mouth_audit,
        },
    )


def compute_metrics(
    y_true,
    probabilities,
    threshold=DECISION_THRESHOLD,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    if (
        len(y_true)
        != len(probabilities)
    ):
        raise ValueError(
            "y_true and probabilities have different lengths."
        )

    if not (
        np.isfinite(
            probabilities
        )
        .all()
    ):
        raise ValueError(
            "NaN/Inf found in probabilities."
        )

    predicted = (
        probabilities
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            predicted,
            labels=[0, 1],
        )
        .ravel()
    )

    return {
        "n": int(
            len(y_true)
        ),
        "threshold": float(
            threshold
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predicted,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predicted,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (
                tn + fp
            )
            if (
                tn + fp
            )
            else np.nan
        ),
        "f1": float(
            f1_score(
                y_true,
                predicted,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier_score": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "tn": int(
            tn
        ),
        "fp": int(
            fp
        ),
        "fn": int(
            fn
        ),
        "tp": int(
            tp
        ),
    }

In [5]:
# ============================================================
# 4) FAST VALIDATION PREDICTION DISCOVERY
# ============================================================

def is_validation_prediction_file(path: Path) -> bool:
    """
    Gerçek validation prediction CSV'si mi kontrol eder.
    TEST dosyalarını kesinlikle kabul etmez.
    """

    path = Path(path)

    if not path.is_file():
        return False

    name = path.name.lower()
    full_text = str(path).lower()

    if path.suffix.lower() != ".csv":
        return False

    if "pred" not in name:
        return False

    # TEST kesinlikle yasak
    if "test" in name:
        return False

    # validation / val anahtarı aranır
    validation_match = (
        "validation" in name
        or re.search(
            r"(^|[_\-.])val([_\-.]|$)",
            name,
        )
        is not None
    )

    if not validation_match:
        return False

    # Video-level yerine frame/image-level tercih edilir,
    # ama bu fonksiyon sadece geçerli adaylığı kontrol eder.
    return True


def get_experiment_root_from_test_prediction(
    test_prediction_path: Path,
) -> Path:
    """
    TEST prediction dosyasından ilgili deney root'unu çıkarır.

    Örnek:
        experiment/predictions/test_predictions.csv

    -> experiment

    Texture modelde:
        experiment/full/predictions/test_predictions.csv

    -> önce full, gerekirse üst root sonraki fonksiyonda ayrıca kontrol edilir.
    """

    test_prediction_path = Path(
        test_prediction_path
    )

    if not test_prediction_path.is_file():
        raise FileNotFoundError(
            f"TEST prediction file not found:\n"
            f"{test_prediction_path}"
        )

    if (
        test_prediction_path
        .parent
        .name
        .lower()
        == "predictions"
    ):
        return (
            test_prediction_path
            .parent
            .parent
        )

    return (
        test_prediction_path
        .parent
    )


def candidate_validation_directories(
    test_prediction_path: Path,
):
    """
    Sadece ilgili deney çevresindeki mantıklı prediction klasörlerine bakar.
    Recursive tüm Drive taraması YAPMAZ.
    """

    test_prediction_path = Path(
        test_prediction_path
    )

    experiment_root = (
        get_experiment_root_from_test_prediction(
            test_prediction_path
        )
    )

    candidates = []

    # Aynı predictions klasörü
    candidates.append(
        test_prediction_path.parent
    )

    # Experiment root / predictions
    candidates.append(
        experiment_root
        / "predictions"
    )

    # Bazı texture deneylerinde full/predictions yapısı var
    candidates.append(
        experiment_root
        / "full"
        / "predictions"
    )

    # Eğer experiment_root zaten "full" ise bir üst experiment de kontrol edilir
    if (
        experiment_root
        .name
        .lower()
        == "full"
    ):
        parent_root = (
            experiment_root
            .parent
        )

        candidates.append(
            parent_root
            / "predictions"
        )

        candidates.append(
            parent_root
            / "full"
            / "predictions"
        )

    # Duplicate temizle
    unique_candidates = []

    for directory in candidates:

        directory = Path(
            directory
        )

        if directory not in unique_candidates:
            unique_candidates.append(
                directory
            )

    return unique_candidates


def score_validation_prediction(
    path: Path,
):
    """
    Birden fazla validation prediction varsa en uygun olanı seç.
    """

    path = Path(
        path
    )

    name = (
        path.name
        .lower()
    )

    score = 0

    # En güçlü tercih
    if "validation" in name:
        score += 100

    if re.search(
        r"(^|[_\-.])val([_\-.]|$)",
        name,
    ):
        score += 90

    # Fusion frame-level olduğu için frame/image tercih edilir
    if "frame" in name:
        score += 30

    if "image" in name:
        score += 20

    # Video-level istemiyoruz
    if "video" in name:
        score -= 50

    # Prediction kelimesi
    if "prediction" in name:
        score += 10

    return score


def discover_validation_prediction(
    family,
    region,
    test_prediction_path,
):
    """
    Validation prediction yolunu hızlı ve güvenli şekilde bulur.

    Öncelik:
    1) Manual override
    2) İlgili predictions klasörlerinde doğrudan dosya taraması
    3) Bulunamazsa None

    TEST prediction hiçbir durumda validation olarak kullanılmaz.
    """

    override = (
        VALIDATION_PREDICTION_OVERRIDE[
            family
        ][region]
    )

    # ========================================================
    # 1) MANUAL OVERRIDE
    # ========================================================

    if override is not None:

        override_path = Path(
            override
        )

        if not override_path.is_file():
            raise FileNotFoundError(
                f"{family}/{region}: "
                f"validation override not found:\n"
                f"{override_path}"
            )

        if (
            "test"
            in override_path.name.lower()
        ):
            raise RuntimeError(
                f"{family}/{region}: "
                "validation override appears to be a TEST file."
            )

        return {
            "path": override_path,
            "discovery": "manual_override",
            "candidate_count": 1,
            "searched_directories": [],
        }


    # ========================================================
    # 2) TARGETED SEARCH
    # ========================================================

    directories = (
        candidate_validation_directories(
            Path(
                test_prediction_path
            )
        )
    )

    candidates = []

    for directory in directories:

        if not directory.is_dir():
            continue

        # DİKKAT:
        # rglob YOK.
        # Sadece klasörün kendisine bakıyoruz.
        for candidate in directory.glob(
            "*.csv"
        ):

            if (
                is_validation_prediction_file(
                    candidate
                )
            ):
                candidates.append(
                    candidate
                )


    # ========================================================
    # 3) SORT
    # ========================================================

    candidates = sorted(
        set(candidates),
        key=lambda path: (
            score_validation_prediction(
                path
            ),
            -len(
                str(path)
            ),
        ),
        reverse=True,
    )


    # ========================================================
    # 4) NOT FOUND
    # ========================================================

    if not candidates:

        return {
            "path": None,
            "discovery": "not_found",
            "candidate_count": 0,
            "searched_directories": [
                str(directory)
                for directory
                in directories
            ],
        }


    # ========================================================
    # 5) FOUND
    # ========================================================

    return {
        "path": candidates[0],
        "discovery": "automatic_targeted",
        "candidate_count": len(
            candidates
        ),
        "alternatives": [
            str(path)
            for path
            in candidates[:10]
        ],
        "searched_directories": [
            str(directory)
            for directory
            in directories
        ],
    }


# ============================================================
# DISCOVER ALL VALIDATION FILES
# ============================================================

VALIDATION_PATHS = {}

validation_discovery_rows = []


for family, cfg in MODEL_FAMILIES.items():

    print(
        "\n"
        + "=" * 90
    )

    print(
        f"VALIDATION DISCOVERY: {family}"
    )

    print(
        "=" * 90
    )

    VALIDATION_PATHS[
        family
    ] = {}


    for region in [
        "eye",
        "brow",
        "mouth",
    ]:

        test_path = (
            cfg[
                f"{region}_test"
            ]
        )

        record = (
            discover_validation_prediction(
                family=family,
                region=region,
                test_prediction_path=test_path,
            )
        )


        VALIDATION_PATHS[
            family
        ][region] = (
            record[
                "path"
            ]
        )


        validation_path = (
            str(
                record[
                    "path"
                ]
            )
            if record[
                "path"
            ]
            is not None
            else None
        )


        print(
            f"{region.upper():5s} | "
            f"{record['discovery']:20s} | "
            f"{validation_path}"
        )


        validation_discovery_rows.append(
            {
                "model_family": family,
                "region": region,

                "test_prediction_path": str(
                    test_path
                ),

                "validation_prediction_path": (
                    validation_path
                ),

                "discovery": (
                    record[
                        "discovery"
                    ]
                ),

                "candidate_count": int(
                    record[
                        "candidate_count"
                    ]
                ),

                "searched_directories": (
                    " | ".join(
                        record.get(
                            "searched_directories",
                            [],
                        )
                    )
                ),
            }
        )


# ============================================================
# SUMMARY
# ============================================================

validation_discovery = (
    pd.DataFrame(
        validation_discovery_rows
    )
)


atomic_write_csv(
    validation_discovery,
    RUN_DIR
    / "audit"
    / "validation_prediction_discovery.csv",
)


print(
    "\n"
    + "=" * 90
)

print(
    "VALIDATION DISCOVERY SUMMARY"
)

print(
    "=" * 90
)


display(
    validation_discovery[
        [
            "model_family",
            "region",
            "validation_prediction_path",
            "discovery",
            "candidate_count",
        ]
    ]
)


VALIDATION DISCOVERY: swinv2_tiny
EYE   | not_found            | None
BROW  | not_found            | None
MOUTH | not_found            | None

VALIDATION DISCOVERY: efficientnet_b0
EYE   | not_found            | None
BROW  | not_found            | None
MOUTH | not_found            | None

VALIDATION DISCOVERY: swinv2_texture
EYE   | not_found            | None
BROW  | not_found            | None
MOUTH | not_found            | None

VALIDATION DISCOVERY SUMMARY


,model_family,region,validation_prediction_path,discovery,candidate_count
0,swinv2_tiny,eye,None,not_found,0
1,swinv2_tiny,brow,None,not_found,0
2,swinv2_tiny,mouth,None,not_found,0
3,efficientnet_b0,eye,None,not_found,0
4,efficientnet_b0,brow,None,not_found,0
5,efficientnet_b0,mouth,None,not_found,0
6,swinv2_texture,eye,None,not_found,0
7,swinv2_texture,brow,None,not_found,0
8,swinv2_texture,mouth,None,not_found,0


In [ ]:
# ============================================================
# 5) PRE-FLIGHT QUALITY GATES
# ============================================================

EXPECTED_KEY_CASES = {
    "fake_test_00000__face_00.jpg": "fake_test_00000",
    "fake_test_00000.jpg": "fake_test_00000",
    "fake_test_fake_test_00000_face00.png": "fake_test_00000",
    "real_test_00125__face_00.jpg": "real_test_00125",
    "real_test_00125.jpg": "real_test_00125",
    "real_test_real_test_00125_face00.png": "real_test_00125",
    "fake_val_00007__face_00.jpg": "fake_val_00007",
    "real_validation_00009.png": "real_val_00009",
}

for raw_key, expected_key in (
    EXPECTED_KEY_CASES.items()
):
    actual_key = (
        canonical_frame_key(
            raw_key
        )
    )

    if actual_key != expected_key:
        raise RuntimeError(
            "Frame-key quality gate failed: "
            f"{raw_key} -> {actual_key}; "
            f"expected {expected_key}"
        )

print(
    "Frame-key normalization: PASSED"
)


# ------------------------------------------------------------
# Test prediction path existence
# ------------------------------------------------------------

test_path_rows = []

for family, cfg in (
    MODEL_FAMILIES.items()
):
    for region in [
        "eye",
        "brow",
        "mouth",
    ]:
        path = Path(
            cfg[
                f"{region}_test"
            ]
        )

        test_path_rows.append(
            {
                "model_family": family,
                "region": region,
                "path": str(
                    path
                ),
                "exists": bool(
                    path.is_file()
                ),
            }
        )


test_path_audit = pd.DataFrame(
    test_path_rows
)

if not (
    test_path_audit[
        "exists"
    ]
    .all()
):
    missing = (
        test_path_audit.loc[
            ~test_path_audit[
                "exists"
            ],
            [
                "model_family",
                "region",
                "path",
            ],
        ]
    )

    raise FileNotFoundError(
        "Configured TEST prediction file(s) missing:\n"
        + missing.to_string(
            index=False
        )
    )

atomic_write_csv(
    test_path_audit,
    RUN_DIR
    / "audit"
    / "configured_test_prediction_paths.csv",
)

print(
    "TEST prediction paths: PASSED"
)


# ------------------------------------------------------------
# Validation prediction gate
# ------------------------------------------------------------

missing_validation = (
    validation_discovery[
        "validation_prediction_path"
    ]
    .isna()
)

if (
    missing_validation
    .any()
):
    missing_table = (
        validation_discovery.loc[
            missing_validation,
            [
                "model_family",
                "region",
                "discovery",
            ],
        ]
    )

    blocking_report = {
        "status": (
            "BLOCKED_MISSING_VALIDATION_PREDICTIONS"
        ),
        "reason": (
            "Learnable Logistic Fusion requires per-sample validation "
            "probabilities for Eye, Brow and Mouth. TEST predictions "
            "cannot be used to train the meta-model."
        ),
        "missing": (
            missing_table
            .to_dict(
                orient="records"
            )
        ),
        "required_action": (
            "Generate/save validation prediction CSVs from the frozen "
            "base models or set verified validation CSV paths in "
            "VALIDATION_PREDICTION_OVERRIDE."
        ),
    }

    atomic_write_json(
        blocking_report,
        RUN_DIR
        / "audit"
        / "BLOCKING_REPORT.json",
    )

    raise RuntimeError(
        "\nLEARNABLE FUSION BLOCKED — "
        "VALIDATION PREDICTIONS ARE MISSING.\n\n"
        + missing_table.to_string(
            index=False
        )
        + "\n\nNo TEST data was used for fusion training. "
        "See audit/BLOCKING_REPORT.json."
    )


# ------------------------------------------------------------
# Explicit no-test-as-validation gate
# ------------------------------------------------------------

for family in MODEL_FAMILIES:
    for region in [
        "eye",
        "brow",
        "mouth",
    ]:
        validation_path = Path(
            VALIDATION_PATHS[
                family
            ][region]
        )

        test_path = Path(
            MODEL_FAMILIES[
                family
            ][
                f"{region}_test"
            ]
        )

        if (
            validation_path.resolve()
            == test_path.resolve()
        ):
            raise RuntimeError(
                f"{family}/{region}: validation and TEST prediction "
                "paths are identical. Data leakage prevented."
            )

        validation_text = (
            str(validation_path)
            .lower()
        )

        if (
            "test_predictions"
            in validation_path.name.lower()
        ):
            raise RuntimeError(
                f"{family}/{region}: selected validation file looks like TEST."
            )

print(
    "Validation prediction isolation: PASSED"
)

In [ ]:
# ============================================================
# 6) LOGISTIC META-MODEL + OOF HELPERS
# ============================================================

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMNS = [
    "p_eye",
    "p_brow",
    "p_mouth",
]


def build_fusion_pipeline():
    """
    Fixed, pre-declared meta-model.
    No test-driven hyperparameter tuning.
    """

    return Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "logistic_regression",
                LogisticRegression(
                    C=LOGISTIC_C,
                    class_weight=(
                        LOGISTIC_CLASS_WEIGHT
                    ),
                    max_iter=(
                        LOGISTIC_MAX_ITER
                    ),
                    random_state=SEED,
                    solver="lbfgs",
                ),
            ),
        ]
    )


def choose_oof_splits(
    y_true,
    desired_splits=5,
):
    class_counts = (
        pd.Series(
            y_true
        )
        .value_counts()
    )

    if len(
        class_counts
    ) != 2:
        raise ValueError(
            "Meta-training validation set must contain both classes."
        )

    minority_count = int(
        class_counts
        .min()
    )

    n_splits = min(
        desired_splits,
        minority_count,
    )

    if n_splits < 2:
        raise RuntimeError(
            "Not enough validation samples per class for OOF evaluation."
        )

    return n_splits


def generate_oof_predictions(
    X,
    y,
):
    X = np.asarray(
        X,
        dtype=float,
    )

    y = np.asarray(
        y,
        dtype=int,
    )

    n_splits = (
        choose_oof_splits(
            y
        )
    )

    splitter = (
        StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=SEED,
        )
    )

    oof_probability = (
        np.full(
            shape=len(y),
            fill_value=np.nan,
            dtype=float,
        )
    )

    fold_records = []

    for fold_index, (
        train_index,
        holdout_index,
    ) in enumerate(
        splitter.split(
            X,
            y,
        ),
        start=1,
    ):
        model = (
            build_fusion_pipeline()
        )

        model.fit(
            X[
                train_index
            ],
            y[
                train_index
            ],
        )

        fold_probability = (
            model.predict_proba(
                X[
                    holdout_index
                ]
            )[:, 1]
        )

        oof_probability[
            holdout_index
        ] = (
            fold_probability
        )

        fold_metrics = (
            compute_metrics(
                y[
                    holdout_index
                ],
                fold_probability,
                threshold=(
                    DECISION_THRESHOLD
                ),
            )
        )

        fold_records.append(
            {
                "fold": (
                    fold_index
                ),
                "train_n": int(
                    len(
                        train_index
                    )
                ),
                "holdout_n": int(
                    len(
                        holdout_index
                    )
                ),
                **fold_metrics,
            }
        )

    if not (
        np.isfinite(
            oof_probability
        )
        .all()
    ):
        raise RuntimeError(
            "OOF probabilities contain NaN/Inf."
        )

    return (
        oof_probability,
        pd.DataFrame(
            fold_records
        ),
        n_splits,
    )


def extract_logistic_parameters(
    fitted_pipeline,
):
    scaler = (
        fitted_pipeline.named_steps[
            "scaler"
        ]
    )

    logistic = (
        fitted_pipeline.named_steps[
            "logistic_regression"
        ]
    )

    coefficients = (
        logistic
        .coef_[0]
    )

    parameter_table = pd.DataFrame(
        {
            "feature": (
                FEATURE_COLUMNS
            ),
            "standardized_coefficient": (
                coefficients
            ),
            "odds_ratio_per_1sd": (
                np.exp(
                    coefficients
                )
            ),
            "scaler_mean": (
                scaler.mean_
            ),
            "scaler_scale": (
                scaler.scale_
            ),
        }
    )

    intercept = float(
        logistic.intercept_[0]
    )

    return (
        parameter_table,
        intercept,
    )

In [ ]:
# ============================================================
# 7) VISUALIZATION HELPERS
# ============================================================

from sklearn.calibration import calibration_curve

DISPLAY_NAMES = {
    "eye_only": "Eye only",
    "brow_only": "Brow only",
    "mouth_only": "Mouth only",
    "fusion": "Learnable Logistic Fusion",
}

PROBABILITY_COLUMNS = {
    "eye_only": "p_eye",
    "brow_only": "p_brow",
    "mouth_only": "p_mouth",
    "fusion": "fusion_probability",
}


def plot_coefficients(
    parameter_table,
    family_name,
    figure_dir,
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    bars = ax.bar(
        parameter_table[
            "feature"
        ],
        parameter_table[
            "standardized_coefficient"
        ],
    )

    ax.axhline(
        0,
        linestyle="--",
        linewidth=1.5,
    )

    ax.set_xticks(
        range(3),
        labels=[
            "Eye",
            "Brow",
            "Mouth",
        ],
    )

    ax.set_title(
        f"{family_name} — Learned Standardized Coefficients",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Logistic Coefficient",
        fontsize=11,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, value in zip(
        bars,
        parameter_table[
            "standardized_coefficient"
        ],
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va=(
                "bottom"
                if value >= 0
                else "top"
            ),
            fontsize=10,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "learned_standardized_coefficients",
    )


def plot_odds_ratios(
    parameter_table,
    family_name,
    figure_dir,
):
    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    values = (
        parameter_table[
            "odds_ratio_per_1sd"
        ]
    )

    bars = ax.bar(
        [
            "Eye",
            "Brow",
            "Mouth",
        ],
        values,
    )

    ax.axhline(
        1.0,
        linestyle="--",
        linewidth=1.5,
        label="Neutral Odds Ratio = 1",
    )

    ax.set_title(
        f"{family_name} — Region Odds Ratios",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Odds Ratio per +1 SD",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, value in zip(
        bars,
        values,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "region_odds_ratios",
    )


def plot_coverage(
    audit,
    family_name,
    split_name,
    figure_dir,
):
    counts = audit[
        "counts"
    ]

    labels = [
        "Eye",
        "Brow",
        "Mouth",
        "Common",
    ]

    values = [
        counts[
            "eye_frames"
        ],
        counts[
            "brow_frames"
        ],
        counts[
            "mouth_frames"
        ],
        counts[
            "common_frames"
        ],
    ]

    fig, ax = plt.subplots(
        figsize=(9, 6)
    )

    bars = ax.bar(
        labels,
        values,
    )

    ax.set_title(
        f"{family_name} — {split_name} Frame Coverage",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Number of Upstream Frames",
        fontsize=11,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    for bar, value in zip(
        bars,
        values,
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            str(
                value
            ),
            ha="center",
            va="bottom",
            fontsize=11,
        )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / f"{split_name.lower()}_frame_coverage",
    )


def plot_roc_comparison(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = (
        dataframe[
            "label"
        ]
        .to_numpy()
    )

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for evaluation, column in (
        PROBABILITY_COLUMNS.items()
    ):
        probability = (
            dataframe[
                column
            ]
            .to_numpy()
        )

        fpr, tpr, _ = (
            roc_curve(
                y_true,
                probability,
            )
        )

        auc_value = (
            roc_auc_score(
                y_true,
                probability,
            )
        )

        ax.plot(
            fpr,
            tpr,
            linewidth=2,
            label=(
                f"{DISPLAY_NAMES[evaluation]} "
                f"(AUC={auc_value:.3f})"
            ),
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Chance",
    )

    ax.set_title(
        f"{family_name} — Test ROC Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "False Positive Rate",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_roc_comparison",
    )


def plot_pr_comparison(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = (
        dataframe[
            "label"
        ]
        .to_numpy()
    )

    fig, ax = plt.subplots(
        figsize=(9, 7)
    )

    for evaluation, column in (
        PROBABILITY_COLUMNS.items()
    ):
        probability = (
            dataframe[
                column
            ]
            .to_numpy()
        )

        precision, recall, _ = (
            precision_recall_curve(
                y_true,
                probability,
            )
        )

        ap_value = (
            average_precision_score(
                y_true,
                probability,
            )
        )

        ax.plot(
            recall,
            precision,
            linewidth=2,
            label=(
                f"{DISPLAY_NAMES[evaluation]} "
                f"(AP={ap_value:.3f})"
            ),
        )

    ax.set_title(
        f"{family_name} — Test Precision–Recall Comparison",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Recall",
        fontsize=11,
    )

    ax.set_ylabel(
        "Precision",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_precision_recall_comparison",
    )


def plot_confusion_matrix(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = (
        dataframe[
            "label"
        ]
        .to_numpy()
    )

    predicted = (
        dataframe[
            "fusion_probability"
        ]
        .to_numpy()
        >= DECISION_THRESHOLD
    ).astype(int)

    matrix = (
        confusion_matrix(
            y_true,
            predicted,
            labels=[0, 1],
        )
    )

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    image = ax.imshow(
        matrix
    )

    ax.set_xticks(
        [0, 1],
        labels=[
            "REAL",
            "FAKE",
        ],
    )

    ax.set_yticks(
        [0, 1],
        labels=[
            "REAL",
            "FAKE",
        ],
    )

    ax.set_xlabel(
        "Predicted Class",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Class",
        fontsize=11,
    )

    ax.set_title(
        f"{family_name} — Logistic Fusion Confusion Matrix",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    for row in range(2):
        for column in range(2):
            ax.text(
                column,
                row,
                str(
                    int(
                        matrix[
                            row,
                            column,
                        ]
                    )
                ),
                ha="center",
                va="center",
                fontsize=12,
            )

    fig.colorbar(
        image,
        ax=ax,
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_fusion_confusion_matrix",
    )


def plot_probability_by_class(
    dataframe,
    family_name,
    figure_dir,
):
    real = (
        dataframe.loc[
            dataframe[
                "label"
            ]
            == 0,
            "fusion_probability",
        ]
        .to_numpy()
    )

    fake = (
        dataframe.loc[
            dataframe[
                "label"
            ]
            == 1,
            "fusion_probability",
        ]
        .to_numpy()
    )

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    ax.boxplot(
        [
            real,
            fake,
        ],
        tick_labels=[
            "REAL",
            "FAKE",
        ],
        showmeans=True,
    )

    ax.axhline(
        DECISION_THRESHOLD,
        linestyle="--",
        linewidth=2,
        label=(
            f"Decision Threshold "
            f"({DECISION_THRESHOLD:.2f})"
        ),
    )

    ax.set_title(
        f"{family_name} — Test Fusion Probability by True Class",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_ylabel(
        "Predicted FAKE Probability",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_fusion_probability_by_true_class",
    )


def plot_calibration(
    dataframe,
    family_name,
    figure_dir,
):
    y_true = (
        dataframe[
            "label"
        ]
        .to_numpy()
    )

    probability = (
        dataframe[
            "fusion_probability"
        ]
        .to_numpy()
    )

    fraction_positive, mean_predicted = (
        calibration_curve(
            y_true,
            probability,
            n_bins=10,
            strategy="quantile",
        )
    )

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    ax.plot(
        mean_predicted,
        fraction_positive,
        marker="o",
        linewidth=2,
        label="Logistic Fusion",
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Perfect Calibration",
    )

    ax.set_title(
        f"{family_name} — Test Calibration Curve",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "Mean Predicted Probability",
        fontsize=11,
    )

    ax.set_ylabel(
        "Observed FAKE Fraction",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "test_calibration_curve",
    )


def plot_oof_roc(
    validation_df,
    oof_probability,
    family_name,
    figure_dir,
):
    y_true = (
        validation_df[
            "label"
        ]
        .to_numpy()
    )

    fpr, tpr, _ = (
        roc_curve(
            y_true,
            oof_probability,
        )
    )

    auc_value = (
        roc_auc_score(
            y_true,
            oof_probability,
        )
    )

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    ax.plot(
        fpr,
        tpr,
        linewidth=2,
        label=(
            f"OOF Logistic Fusion "
            f"(AUC={auc_value:.3f})"
        ),
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1.5,
        label="Chance",
    )

    ax.set_title(
        f"{family_name} — Validation OOF ROC",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        "False Positive Rate",
        fontsize=11,
    )

    ax.set_ylabel(
        "True Positive Rate",
        fontsize=11,
    )

    ax.legend(
        frameon=True
    )

    ax.grid(
        alpha=0.25
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "validation_oof_roc",
    )


def plot_metric_comparison(
    metrics_df,
    family_name,
    figure_dir,
):
    selected = (
        metrics_df[
            [
                "evaluation",
                "roc_auc",
                "pr_auc",
                "balanced_accuracy",
                "f1",
            ]
        ]
        .copy()
    )

    selected[
        "evaluation"
    ] = (
        selected[
            "evaluation"
        ]
        .map(
            DISPLAY_NAMES
        )
    )

    plot_df = (
        selected
        .set_index(
            "evaluation"
        )
    )

    fig, ax = plt.subplots(
        figsize=(11, 7)
    )

    plot_df.plot(
        kind="bar",
        ax=ax,
    )

    ax.set_title(
        f"{family_name} — Regional vs Learnable Fusion Metrics",
        fontsize=14,
        fontweight="bold",
        pad=12,
    )

    ax.set_xlabel(
        ""
    )

    ax.set_ylabel(
        "Score",
        fontsize=11,
    )

    ax.set_ylim(
        0,
        1.05,
    )

    ax.tick_params(
        axis="x",
        rotation=20,
    )

    ax.legend(
        [
            "ROC-AUC",
            "PR-AUC",
            "Balanced Accuracy",
            "F1",
        ],
        frameon=True,
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    save_figure(
        fig,
        figure_dir
        / "regional_vs_learnable_fusion_metrics",
    )

In [ ]:
# ============================================================
# 8) LEARNABLE LOGISTIC FUSION — ALL THREE MODEL FAMILIES
# ============================================================

all_metric_rows = []
family_audit_rows = []
coefficient_rows = []
oof_summary_rows = []

for family, cfg in (
    MODEL_FAMILIES.items()
):
    print(
        "\n"
        + "=" * 90
    )

    print(
        f"MODEL FAMILY: {family}"
    )

    print(
        "=" * 90
    )

    # --------------------------------------------------------
    # A) Align meta-training VALIDATION predictions
    # --------------------------------------------------------

    validation_df, validation_audit = (
        align_three(
            VALIDATION_PATHS[
                family
            ][
                "eye"
            ],
            VALIDATION_PATHS[
                family
            ][
                "brow"
            ],
            VALIDATION_PATHS[
                family
            ][
                "mouth"
            ],
        )
    )

    # --------------------------------------------------------
    # B) Align final TEST predictions
    # --------------------------------------------------------

    test_df, test_audit = (
        align_three(
            cfg[
                "eye_test"
            ],
            cfg[
                "brow_test"
            ],
            cfg[
                "mouth_test"
            ],
        )
    )

    # --------------------------------------------------------
    # C) Split leakage quality gate by normalized fusion keys
    # --------------------------------------------------------

    validation_keys = set(
        validation_df[
            "fusion_key"
        ]
    )

    test_keys = set(
        test_df[
            "fusion_key"
        ]
    )

    overlap = (
        validation_keys
        & test_keys
    )

    if overlap:
        overlap_examples = sorted(
            overlap
        )[:10]

        raise RuntimeError(
            f"{family}: validation/test upstream frame overlap "
            f"detected ({len(overlap)} keys).\n"
            f"Examples: {overlap_examples}"
        )

    # --------------------------------------------------------
    # D) Prepare meta-training arrays
    # --------------------------------------------------------

    X_validation = (
        validation_df[
            FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    y_validation = (
        validation_df[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    X_test = (
        test_df[
            FEATURE_COLUMNS
        ]
        .to_numpy(
            dtype=float
        )
    )

    y_test = (
        test_df[
            "label"
        ]
        .to_numpy(
            dtype=int
        )
    )

    # --------------------------------------------------------
    # E) Validation-only OOF sanity check
    # --------------------------------------------------------

    (
        oof_probability,
        oof_fold_metrics,
        oof_n_splits,
    ) = generate_oof_predictions(
        X_validation,
        y_validation,
    )

    validation_df[
        "oof_fusion_probability"
    ] = (
        oof_probability
    )

    oof_metrics = (
        compute_metrics(
            y_validation,
            oof_probability,
            threshold=DECISION_THRESHOLD,
        )
    )

    oof_summary_rows.append(
        {
            "model_family": family,
            "oof_n_splits": int(
                oof_n_splits
            ),
            **oof_metrics,
        }
    )

    # --------------------------------------------------------
    # F) Fit FINAL fusion model on ALL aligned validation data
    # --------------------------------------------------------

    fusion_model = (
        build_fusion_pipeline()
    )

    fusion_model.fit(
        X_validation,
        y_validation,
    )

    # --------------------------------------------------------
    # G) Final TEST inference only
    # --------------------------------------------------------

    test_probability = (
        fusion_model
        .predict_proba(
            X_test
        )[:, 1]
    )

    if not (
        np.isfinite(
            test_probability
        )
        .all()
    ):
        raise RuntimeError(
            f"{family}: TEST fusion probability contains NaN/Inf."
        )

    if (
        (
            (test_probability < 0)
            | (test_probability > 1)
        )
        .any()
    ):
        raise RuntimeError(
            f"{family}: TEST fusion probability outside [0, 1]."
        )

    test_df[
        "fusion_probability"
    ] = (
        test_probability
    )

    # --------------------------------------------------------
    # H) Explainability parameters
    # --------------------------------------------------------

    (
        parameter_table,
        intercept,
    ) = (
        extract_logistic_parameters(
            fusion_model
        )
    )

    parameter_table.insert(
        0,
        "model_family",
        family,
    )

    parameter_table[
        "intercept"
    ] = (
        intercept
    )

    coefficient_rows.extend(
        parameter_table
        .to_dict(
            orient="records"
        )
    )

    # --------------------------------------------------------
    # I) Metrics on the SAME aligned TEST set
    # --------------------------------------------------------

    probability_columns = {
        "eye_only": "p_eye",
        "brow_only": "p_brow",
        "mouth_only": "p_mouth",
        "fusion": "fusion_probability",
    }

    metric_rows = []

    for (
        evaluation,
        probability_column,
    ) in probability_columns.items():

        metrics = (
            compute_metrics(
                y_test,
                test_df[
                    probability_column
                ],
                threshold=(
                    DECISION_THRESHOLD
                ),
            )
        )

        row = {
            "model_family": family,
            "method": METHOD_NAME,
            "evaluation": evaluation,
            "threshold_source": (
                "fixed_predefined_0.50"
            ),
            "meta_training_source": (
                "aligned_validation_predictions"
            ),
            "logistic_C": (
                LOGISTIC_C
            ),
            "logistic_class_weight": (
                LOGISTIC_CLASS_WEIGHT
            ),
            **metrics,
        }

        metric_rows.append(
            row
        )

        all_metric_rows.append(
            row
        )

    family_metrics = pd.DataFrame(
        metric_rows
    )

    # --------------------------------------------------------
    # J) Save outputs atomically
    # --------------------------------------------------------

    family_dir = (
        RUN_DIR
        / family
    )

    figure_dir = (
        family_dir
        / "figures"
    )

    atomic_dump_joblib(
        fusion_model,
        family_dir
        / "model"
        / "fusion_logistic_regression.joblib",
    )

    # Reload and verify same probability for first sample.
    reloaded_model = joblib.load(
        family_dir
        / "model"
        / "fusion_logistic_regression.joblib"
    )

    original_check = float(
        fusion_model.predict_proba(
            X_test[
                :1
            ]
        )[0, 1]
    )

    reloaded_check = float(
        reloaded_model.predict_proba(
            X_test[
                :1
            ]
        )[0, 1]
    )

    if not math.isclose(
        original_check,
        reloaded_check,
        rel_tol=1e-12,
        abs_tol=1e-12,
    ):
        raise RuntimeError(
            f"{family}: saved/reloaded model inference mismatch."
        )

    atomic_write_csv(
        validation_df,
        family_dir
        / "predictions"
        / "aligned_validation_predictions_with_oof.csv",
    )

    atomic_write_csv(
        test_df,
        family_dir
        / "predictions"
        / "aligned_test_predictions.csv",
    )

    atomic_write_csv(
        family_metrics,
        family_dir
        / "metrics"
        / "test_metrics.csv",
    )

    atomic_write_csv(
        oof_fold_metrics,
        family_dir
        / "metrics"
        / "validation_oof_fold_metrics.csv",
    )

    atomic_write_csv(
        parameter_table,
        family_dir
        / "metrics"
        / "logistic_parameters.csv",
    )

    atomic_write_json(
        {
            "run_id": RUN_ID,
            "method": METHOD_NAME,
            "model_family": family,
            "decision_threshold": (
                DECISION_THRESHOLD
            ),
            "threshold_source": (
                "fixed_predefined_0.50"
            ),
            "logistic_hyperparameters": {
                "C": LOGISTIC_C,
                "class_weight": (
                    LOGISTIC_CLASS_WEIGHT
                ),
                "max_iter": (
                    LOGISTIC_MAX_ITER
                ),
                "solver": "lbfgs",
                "random_state": SEED,
            },
            "meta_training": {
                "source": (
                    "aligned validation predictions"
                ),
                "n": int(
                    len(
                        validation_df
                    )
                ),
                "oof_splits": int(
                    oof_n_splits
                ),
                "oof_metrics": (
                    oof_metrics
                ),
            },
            "test_evaluation": {
                "n": int(
                    len(
                        test_df
                    )
                ),
                "used_for_training": False,
                "used_for_hyperparameter_selection": False,
            },
            "validation_alignment_audit": (
                validation_audit
            ),
            "test_alignment_audit": (
                test_audit
            ),
            "standardized_coefficients": {
                row["feature"]: float(
                    row[
                        "standardized_coefficient"
                    ]
                )
                for row in (
                    parameter_table
                    .to_dict(
                        orient="records"
                    )
                )
            },
            "intercept": (
                intercept
            ),
            "methodological_limitation": (
                "Base models may already have used their validation split "
                "for early stopping/model selection. A dedicated meta-training "
                "split or OOF base predictions would be stronger."
            ),
        },
        family_dir
        / "audit"
        / "run_audit.json",
    )

    # --------------------------------------------------------
    # K) Visuals
    # --------------------------------------------------------

    figure_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    plot_coefficients(
        parameter_table=(
            parameter_table
        ),
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_odds_ratios(
        parameter_table=(
            parameter_table
        ),
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_coverage(
        audit=validation_audit,
        family_name=family,
        split_name="Validation",
        figure_dir=figure_dir,
    )

    plot_coverage(
        audit=test_audit,
        family_name=family,
        split_name="Test",
        figure_dir=figure_dir,
    )

    plot_oof_roc(
        validation_df=(
            validation_df
        ),
        oof_probability=(
            oof_probability
        ),
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_roc_comparison(
        dataframe=test_df,
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_pr_comparison(
        dataframe=test_df,
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_confusion_matrix(
        dataframe=test_df,
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_probability_by_class(
        dataframe=test_df,
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_calibration(
        dataframe=test_df,
        family_name=family,
        figure_dir=figure_dir,
    )

    plot_metric_comparison(
        metrics_df=family_metrics,
        family_name=family,
        figure_dir=figure_dir,
    )

    # --------------------------------------------------------
    # L) Family audit summary
    # --------------------------------------------------------

    family_audit_rows.append(
        {
            "model_family": family,
            "validation_common_frames": int(
                validation_audit[
                    "counts"
                ][
                    "common_frames"
                ]
            ),
            "validation_real": int(
                validation_audit[
                    "counts"
                ][
                    "real_common_frames"
                ]
            ),
            "validation_fake": int(
                validation_audit[
                    "counts"
                ][
                    "fake_common_frames"
                ]
            ),
            "test_common_frames": int(
                test_audit[
                    "counts"
                ][
                    "common_frames"
                ]
            ),
            "test_real": int(
                test_audit[
                    "counts"
                ][
                    "real_common_frames"
                ]
            ),
            "test_fake": int(
                test_audit[
                    "counts"
                ][
                    "fake_common_frames"
                ]
            ),
            "oof_n_splits": int(
                oof_n_splits
            ),
            "oof_roc_auc": float(
                oof_metrics[
                    "roc_auc"
                ]
            ),
            "test_fusion_roc_auc": float(
                family_metrics.loc[
                    family_metrics[
                        "evaluation"
                    ]
                    == "fusion",
                    "roc_auc",
                ]
                .iloc[0]
            ),
        }
    )

    print(
        "Validation common frames:",
        validation_audit[
            "counts"
        ][
            "common_frames"
        ],
    )

    print(
        "Test common frames:",
        test_audit[
            "counts"
        ][
            "common_frames"
        ],
    )

    print(
        "OOF ROC-AUC:",
        f"{oof_metrics['roc_auc']:.4f}",
    )

    print(
        "Standardized coefficients:"
    )

    display(
        parameter_table[
            [
                "feature",
                "standardized_coefficient",
                "odds_ratio_per_1sd",
            ]
        ]
    )

    display(
        family_metrics[
            [
                "evaluation",
                "n",
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall",
                "specificity",
                "f1",
                "roc_auc",
                "pr_auc",
                "brier_score",
            ]
        ]
    )


all_metrics = pd.DataFrame(
    all_metric_rows
)

family_audit = pd.DataFrame(
    family_audit_rows
)

all_coefficients = pd.DataFrame(
    coefficient_rows
)

oof_summary = pd.DataFrame(
    oof_summary_rows
)

atomic_write_csv(
    all_metrics,
    RUN_DIR
    / "metrics"
    / "all_model_families_metrics.csv",
)

atomic_write_csv(
    family_audit,
    RUN_DIR
    / "audit"
    / "family_alignment_summary.csv",
)

atomic_write_csv(
    all_coefficients,
    RUN_DIR
    / "metrics"
    / "all_logistic_coefficients.csv",
)

atomic_write_csv(
    oof_summary,
    RUN_DIR
    / "metrics"
    / "all_validation_oof_metrics.csv",
)

print(
    "\nAll three model families completed."
)

In [ ]:
# ============================================================
# 9) CROSS-FAMILY VISUAL COMPARISONS
# ============================================================

overall_figure_dir = (
    RUN_DIR
    / "figures"
)

overall_figure_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# A) Final fusion metrics across model families
# ------------------------------------------------------------

fusion_only = (
    all_metrics.loc[
        all_metrics[
            "evaluation"
        ]
        == "fusion"
    ]
    .copy()
    .sort_values(
        "model_family"
    )
)

metric_columns = [
    "roc_auc",
    "pr_auc",
    "balanced_accuracy",
    "f1",
]

metric_plot = (
    fusion_only[
        [
            "model_family",
            *metric_columns,
        ]
    ]
    .set_index(
        "model_family"
    )
)

fig, ax = plt.subplots(
    figsize=(12, 7)
)

metric_plot.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Learnable Logistic Fusion — Model Family Comparison",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Score",
    fontsize=11,
)

ax.set_ylim(
    0,
    1.05,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "ROC-AUC",
        "PR-AUC",
        "Balanced Accuracy",
        "F1",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "learnable_logistic_model_family_comparison",
)


# ------------------------------------------------------------
# B) Learned coefficients across model families
# ------------------------------------------------------------

coefficient_pivot = (
    all_coefficients
    .pivot(
        index="model_family",
        columns="feature",
        values="standardized_coefficient",
    )
    .rename(
        columns={
            "p_eye": "Eye",
            "p_brow": "Brow",
            "p_mouth": "Mouth",
        }
    )
)

fig, ax = plt.subplots(
    figsize=(12, 7)
)

coefficient_pivot.plot(
    kind="bar",
    ax=ax,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
)

ax.set_title(
    "Learned Region Coefficients Across Model Families",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Standardized Logistic Coefficient",
    fontsize=11,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    frameon=True
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "learned_region_coefficients_across_families",
)


# ------------------------------------------------------------
# C) OOF validation AUC vs final test AUC
# ------------------------------------------------------------

generalization_plot = (
    family_audit[
        [
            "model_family",
            "oof_roc_auc",
            "test_fusion_roc_auc",
        ]
    ]
    .set_index(
        "model_family"
    )
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

generalization_plot.plot(
    kind="bar",
    ax=ax,
)

ax.set_title(
    "Logistic Fusion — Validation OOF vs Final Test ROC-AUC",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "ROC-AUC",
    fontsize=11,
)

ax.set_ylim(
    0,
    1.05,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "Validation OOF ROC-AUC",
        "Final Test ROC-AUC",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "validation_oof_vs_test_auc",
)


# ------------------------------------------------------------
# D) Gain vs best single region
# ------------------------------------------------------------

gain_rows = []

for family in (
    all_metrics[
        "model_family"
    ]
    .unique()
):
    subset = all_metrics[
        all_metrics[
            "model_family"
        ]
        == family
    ]

    single = subset[
        subset[
            "evaluation"
        ].isin(
            [
                "eye_only",
                "brow_only",
                "mouth_only",
            ]
        )
    ]

    fusion = subset[
        subset[
            "evaluation"
        ]
        == "fusion"
    ].iloc[0]

    gain_rows.append(
        {
            "model_family": family,
            "roc_auc_gain_vs_best_single": float(
                fusion["roc_auc"]
                - single[
                    "roc_auc"
                ].max()
            ),
            "f1_gain_vs_best_single": float(
                fusion["f1"]
                - single[
                    "f1"
                ].max()
            ),
        }
    )

gain_df = pd.DataFrame(
    gain_rows
)

atomic_write_csv(
    gain_df,
    RUN_DIR
    / "metrics"
    / "fusion_gain_vs_best_single_region.csv",
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

gain_df.set_index(
    "model_family"
).plot(
    kind="bar",
    ax=ax,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5,
)

ax.set_title(
    "Learnable Logistic Fusion — Gain vs Best Single Region",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

ax.set_xlabel(
    "Model Family",
    fontsize=11,
)

ax.set_ylabel(
    "Absolute Metric Difference",
    fontsize=11,
)

ax.tick_params(
    axis="x",
    rotation=15,
)

ax.legend(
    [
        "ROC-AUC Gain",
        "F1 Gain",
    ],
    frameon=True,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

fig.tight_layout()

save_figure(
    fig,
    overall_figure_dir
    / "fusion_gain_vs_best_single_region",
)

display(
    fusion_only
)

display(
    gain_df
)

In [ ]:
# ============================================================
# 10) FINAL QUALITY GATES + OUTPUT MANIFEST
# ============================================================

required_metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
    "brier_score",
]

missing_metric_columns = [
    column
    for column in (
        required_metric_columns
    )
    if (
        column
        not in all_metrics.columns
    )
]

if missing_metric_columns:
    raise RuntimeError(
        f"Missing metric columns: "
        f"{missing_metric_columns}"
    )

numeric_metrics = (
    all_metrics[
        required_metric_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

if not (
    np.isfinite(
        numeric_metrics
        .to_numpy()
    )
    .all()
):
    raise RuntimeError(
        "NaN/Inf found in final metrics."
    )

for column in (
    required_metric_columns
):
    invalid = (
        (
            all_metrics[
                column
            ]
            < 0
        )
        | (
            all_metrics[
                column
            ]
            > 1
        )
    )

    if (
        invalid
        .any()
    ):
        raise RuntimeError(
            f"{column}: metric outside [0, 1]."
        )


expected_evaluations = {
    "eye_only",
    "brow_only",
    "mouth_only",
    "fusion",
}

for family in MODEL_FAMILIES:
    actual = set(
        all_metrics.loc[
            all_metrics[
                "model_family"
            ]
            == family,
            "evaluation",
        ]
    )

    if (
        actual
        != expected_evaluations
    ):
        raise RuntimeError(
            f"{family}: expected evaluations "
            f"{expected_evaluations}, found {actual}"
        )


# ------------------------------------------------------------
# Model artifact inference test
# ------------------------------------------------------------

for family in MODEL_FAMILIES:
    model_path = (
        RUN_DIR
        / family
        / "model"
        / "fusion_logistic_regression.joblib"
    )

    if not model_path.is_file():
        raise RuntimeError(
            f"{family}: saved fusion model missing."
        )

    loaded_model = (
        joblib.load(
            model_path
        )
    )

    if not hasattr(
        loaded_model,
        "predict_proba",
    ):
        raise RuntimeError(
            f"{family}: reloaded fusion model has no predict_proba."
        )


# ------------------------------------------------------------
# Figure resolution + output manifest
# ------------------------------------------------------------

manifest_rows = []

for path in sorted(
    RUN_DIR.rglob("*")
):
    if not (
        path.is_file()
    ):
        continue

    record = {
        "relative_path": str(
            path.relative_to(
                RUN_DIR
            )
        ),
        "size_bytes": int(
            path.stat().st_size
        ),
        "suffix": (
            path.suffix
            .lower()
        ),
    }

    if (
        path.suffix.lower()
        == ".png"
    ):
        with Image.open(
            path
        ) as image:
            record[
                "width_px"
            ] = int(
                image.width
            )

            record[
                "height_px"
            ] = int(
                image.height
            )

            if (
                min(
                    image.size
                )
                < MIN_FIGURE_SHORT_EDGE_PX
            ):
                raise RuntimeError(
                    "Figure below minimum pixel size: "
                    f"{path} -> {image.size}"
                )

    manifest_rows.append(
        record
    )


manifest = pd.DataFrame(
    manifest_rows
)

atomic_write_csv(
    manifest,
    RUN_DIR
    / "output_manifest.csv",
)


final_summary = {
    "run_id": RUN_ID,
    "method": METHOD_NAME,
    "seed": SEED,
    "decision_threshold": (
        DECISION_THRESHOLD
    ),
    "threshold_source": (
        "fixed_predefined_0.50"
    ),
    "meta_training_source": (
        "aligned validation predictions only"
    ),
    "test_used_for_training": False,
    "test_used_for_hyperparameter_selection": False,
    "logistic_hyperparameters": {
        "C": LOGISTIC_C,
        "class_weight": (
            LOGISTIC_CLASS_WEIGHT
        ),
        "max_iter": (
            LOGISTIC_MAX_ITER
        ),
        "solver": "lbfgs",
    },
    "frame_aggregation": (
        FRAME_AGGREGATION
    ),
    "model_families_completed": sorted(
        all_metrics[
            "model_family"
        ]
        .unique()
        .tolist()
    ),
    "metric_rows": int(
        len(
            all_metrics
        )
    ),
    "output_files_before_manifest": int(
        len(
            manifest_rows
        )
    ),
    "quality_gates": (
        "PASSED"
    ),
    "methodological_limitation": (
        "Base models may have used validation for early stopping/model "
        "selection. Dedicated meta-training or OOF base predictions would "
        "provide a stronger stacking protocol."
    ),
}

atomic_write_json(
    final_summary,
    RUN_DIR
    / "run_summary.json",
)


print(
    "=" * 90
)

print(
    "FINAL QUALITY GATES: PASSED"
)

print(
    "=" * 90
)

print(
    f"Run ID : {RUN_ID}"
)

print(
    f"Output : {RUN_DIR}"
)

print(
    "PNG figures:",
    int(
        (
            manifest[
                "suffix"
            ]
            == ".png"
        )
        .sum()
    ),
)

print(
    "SVG figures:",
    int(
        (
            manifest[
                "suffix"
            ]
            == ".svg"
        )
        .sum()
    ),
)

display(
    all_metrics.sort_values(
        [
            "model_family",
            "evaluation",
        ]
    )
)

## Output structure

```text
Fusion_Experiments/
└── 03_learnable_logistic_fusion/
    └── <RUN_ID>/
        ├── environment.json
        ├── run_summary.json
        ├── output_manifest.csv
        ├── audit/
        │   ├── validation_prediction_discovery.csv
        │   ├── configured_test_prediction_paths.csv
        │   └── family_alignment_summary.csv
        ├── metrics/
        │   ├── all_model_families_metrics.csv
        │   ├── all_logistic_coefficients.csv
        │   ├── all_validation_oof_metrics.csv
        │   └── fusion_gain_vs_best_single_region.csv
        ├── figures/
        │   ├── learnable_logistic_model_family_comparison.*
        │   ├── learned_region_coefficients_across_families.*
        │   ├── validation_oof_vs_test_auc.*
        │   └── fusion_gain_vs_best_single_region.*
        └── <model_family>/
            ├── model/
            │   └── fusion_logistic_regression.joblib
            ├── audit/
            │   └── run_audit.json
            ├── predictions/
            │   ├── aligned_validation_predictions_with_oof.csv
            │   └── aligned_test_predictions.csv
            ├── metrics/
            │   ├── test_metrics.csv
            │   ├── validation_oof_fold_metrics.csv
            │   └── logistic_parameters.csv
            └── figures/
                ├── learned_standardized_coefficients.*
                ├── region_odds_ratios.*
                ├── validation_frame_coverage.*
                ├── test_frame_coverage.*
                ├── validation_oof_roc.*
                ├── test_roc_comparison.*
                ├── test_precision_recall_comparison.*
                ├── test_fusion_confusion_matrix.*
                ├── test_fusion_probability_by_true_class.*
                ├── test_calibration_curve.*
                └── regional_vs_learnable_fusion_metrics.*
```

`*` = both PNG (600 DPI) and SVG.